# Tashkent Apartment Listing Price Predictor — 2026 demo
This focused demo loads the committed pipeline and estimates an **August 2026 advertised asking price in USD**. It does not claim a completed sale price or legal appraisal.

In [ ]:
from pathlib import Path
import os

if not Path('src').exists():
    os.system('git clone -q https://github.com/DilnuraHamdamova/tashkent-house-price-predictor.git')
    os.chdir('tashkent-house-price-predictor')
    os.system('python -m pip install -q -r requirements.txt')
print('Repository ready:', Path.cwd())

In [ ]:
from src.model import load_artifact, predict_price

artifact = load_artifact('artifacts/house_price_pipeline.joblib')
metadata = artifact['metadata']
print('Model:', metadata['model_name'])
print('Target:', metadata['target'])
print('Training snapshot:', metadata['data_audit']['listing_date_min'], 'to', metadata['data_audit']['listing_date_max'])

## Real input → resale asking-price estimate

In [ ]:
example = {
    'district': 'Chilonzor',
    'size_m2': 70,
    'rooms': 3,
    'level': 3,
    'max_levels': 5,
    'is_new_building': 0,
}
price, warnings = predict_price(artifact, **example)
print(f'Estimated August 2026 resale asking price: ${price:,.0f} USD')
print('Warnings:', warnings or 'None')

## Same apartment marked as a new build

In [ ]:
new_build_price, warnings = predict_price(artifact, **{**example, 'is_new_building': 1})
print(f'Estimated August 2026 new-build asking price: ${new_build_price:,.0f} USD')
print('Warnings:', warnings or 'None')

## Validation edge case

In [ ]:
try:
    predict_price(artifact, **{**example, 'level': 9, 'max_levels': 5})
except ValueError as error:
    print('Expected validation error:', error)

## Protected unseen-data evidence

In [ ]:
test = metadata['protected_test_comparison']
print('Baseline MAE:', f"${test['median_baseline']['mae_usd']:,.0f}")
print('Random Forest MAE:', f"${test['random_forest']['mae_usd']:,.0f}")
print('Random Forest R²:', f"{test['random_forest']['r2']:.3f}")
print('Random Forest MAPE:', f"{test['random_forest']['mape_percent']:.2f}%")